# EDA por partición de campañas — Tarjetas Santander

Análisis exploratorio de la base de clientes de tarjetas (10,000 clientes) **seccionado por las
particiones con las que se diseñan las campañas**: cada campaña apunta a un `segmento_objetivo` × `producto`.
Hay 20 campañas (2024–2025) que se agrupan en **6 particiones**:

| Partición | Segmento objetivo | Producto | Campañas |
|---|---|---|---|
| P1 | Mass | LikeU | 8 |
| P2 | Todos | Todas | 6 |
| P3 | Premier | Aeroméxico Platinum | 2 |
| P4 | Select | Fiesta Rewards | 1 |
| P5 | Select | Platinum | 2 |
| P6 | Premier | World Elite | 1 |

Para cada partición se revisa: (a) desempeño de sus campañas, (b) la población de clientes que le corresponde,
(c) en qué se diferencia ese perfil del resto de la base, (d) distribuciones de las métricas palanca y
(e) bolsas de oportunidad. Al final hay un comparativo entre particiones y conclusiones.

**Contenido**
1. Setup y carga
2. Calidad de datos
3. Definición de particiones y mapeo a clientes
4. Panorama de campañas por partición
5. EDA por partición (P1 … P6)
6. Comparativo entre particiones
7. Conclusiones y siguientes pasos

## 1. Setup y carga

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Render nativo de plotly (VS Code / JupyterLab)
pio.renderers.default = "plotly_mimetype"

# Paleta (validada para daltonismo en pares adyacentes) y tema sobrio
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GRIS = "#a3a29c"
POS, NEG, NEUTRO = "#2a78d6", "#e34948", "#f0efec"
DIVERGENTE = [[0, "#b83a3a"], [0.25, "#e88a89"], [0.5, NEUTRO], [0.75, "#86b6ef"], [1, "#1c5cab"]]
SECUENCIAL = [[0, "#cde2fb"], [0.5, "#3987e5"], [1, "#0d366b"]]

pio.templates["santander"] = go.layout.Template(
    layout=dict(
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#0b0b0b"),
        paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
        colorway=PAL,
        xaxis=dict(gridcolor="#e8e7e3", zerolinecolor="#c3c2b7", linecolor="#c3c2b7"),
        yaxis=dict(gridcolor="#e8e7e3", zerolinecolor="#c3c2b7", linecolor="#c3c2b7"),
        hoverlabel=dict(bgcolor="white", font_size=12),
        margin=dict(l=60, r=30, t=60, b=50),
        bargap=0.25,
    )
)
pio.templates.default = "simple_white+santander"

DATA = "data/dataset_dummy_tarjetas_santander_10000_clientes.csv.xlsx"
xls = pd.ExcelFile(DATA)
print(xls.sheet_names)

['dataset_dummy_tarjetas_santande', 'Glosario Tarjetas', 'Dataset_dummy_Campañas', 'Glosario Campañas']


In [2]:
clientes = pd.read_excel(DATA, sheet_name=0)
campanas = pd.read_excel(DATA, sheet_name=2)
glosario_campanas = pd.read_excel(DATA, sheet_name=3)

# Nombres de columnas sin acentos para trabajar cómodo
campanas = campanas.rename(columns={"año": "anio", "campaña": "campana"})

print("clientes:", clientes.shape, "| campañas:", campanas.shape)
display(clientes.head())
display(campanas.head())

clientes: (10000, 16) | campañas: (20, 10)


,customer_id,edad,segmento,antiguedad_meses,ingreso_mensual,tarjeta_santander,uso_tarjeta_pct,transacciones_mes,gasto_mensual,ecommerce_pct,pagos_puntuales_pct,app_logins_mes,nps,satisfaccion,abandono_riesgo,activacion_promos_pct
0,C000001,56,Mass,15,"37,191.83",LikeU,15.60,19,"22,273.76",97.00,95.00,7,4,3.50,0.61,0.70
1,C000002,42,Premier,59,"20,621.54",Fiesta Rewards Platinum,23.30,11,"10,338.86",98.30,84.00,9,8,4.90,0.81,30.50
2,C000003,70,Mass,92,"16,142.71",Aeromexico Blanca,3.40,25,"10,096.71",31.20,85.60,10,1,4.10,0.94,89.50
3,C000004,31,Mass,15,"25,391.66",Free,27.10,23,"9,359.37",54.30,74.20,13,6,2.70,0.39,29.30
4,C000005,28,Mass,136,"42,073.20",LikeU,35.80,20,"15,932.13",31.10,79.80,12,10,3.20,0.38,97.20


,campaign_id,anio,campana,segmento_objetivo,producto,clientes_objetivo,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct
0,CAM001,2024,Cashback LikeU Farmacias,Mass,LikeU,150000,72,28,12,18
1,CAM002,2024,Cashback LikeU Restaurantes,Mass,LikeU,180000,75,31,15,22
2,CAM003,2024,Cashback LikeU Telecom,Mass,LikeU,120000,68,25,9,14
3,CAM004,2024,Activación Tarjeta Digital,Todos,Todas,250000,81,42,7,11
4,CAM005,2024,Hot Sale MSI,Todos,Todas,500000,84,35,21,27


In [3]:
display(glosario_campanas)

,Campo,Significado
0,campaign_id,Identificador único de campaña
1,año,Año de ejecución
2,campaña,Nombre descriptivo
3,segmento_objetivo,Segmento de clientes objetivo
4,producto,Tarjeta o familia de productos
5,clientes_objetivo,Clientes impactados
6,apertura_pct,% que abrió el mensaje/comunicación
7,conversion_pct,% que realizó la acción esperada
8,incremento_gasto_pct,Incremento promedio del gasto
9,incremento_uso_tarjeta_pct,Incremento promedio en frecuencia de uso


## 2. Calidad de datos

In [4]:
calidad = pd.DataFrame({
    "dtype": clientes.dtypes.astype(str),
    "nulos": clientes.isna().sum(),
    "unicos": clientes.nunique(),
    "min": clientes.min(numeric_only=True),
    "max": clientes.max(numeric_only=True),
})
print("Duplicados de fila:", clientes.duplicated().sum(),
      "| customer_id duplicados:", clientes["customer_id"].duplicated().sum())
display(calidad)

Duplicados de fila: 0 | customer_id duplicados: 0


,dtype,nulos,unicos,min,max
abandono_riesgo,float64,0,101,0.00,1.00
activacion_promos_pct,float64,0,1001,0.00,100.00
antiguedad_meses,int64,0,180,1.00,180.00
app_logins_mes,int64,0,23,1.00,23.00
customer_id,str,0,10000,NaN,NaN
ecommerce_pct,float64,0,1001,0.00,100.00
edad,int64,0,58,18.00,75.00
gasto_mensual,float64,0,9978,687.52,"56,173.87"
ingreso_mensual,float64,0,9992,"4,600.65","193,139.02"
nps,int64,0,11,0.00,10.00


In [5]:
NUM = [c for c in clientes.select_dtypes("number").columns]
display(clientes[NUM].describe(percentiles=[.05, .25, .5, .75, .95]).T)

# Chequeos de rango de negocio
rangos = {
    "uso_tarjeta_pct": (0, 100), "ecommerce_pct": (0, 100), "pagos_puntuales_pct": (0, 100),
    "activacion_promos_pct": (0, 100), "abandono_riesgo": (0, 1), "nps": (0, 10), "satisfaccion": (1, 5),
}
fuera = {c: int((~clientes[c].between(*r)).sum()) for c, r in rangos.items()}
print("Registros fuera de rango:", fuera)
print("Campañas con conversión > apertura:", int((campanas.conversion_pct > campanas.apertura_pct).sum()))

,count,mean,std,min,5%,25%,50%,75%,95%,max
edad,"10,000.00",46.48,16.73,18.00,20.00,32.00,47.00,61.00,72.00,75.00
antiguedad_meses,"10,000.00",91.66,51.59,1.00,10.00,47.00,92.00,135.00,172.00,180.00
ingreso_mensual,"10,000.00","30,627.46","16,263.78","4,600.65","11,941.93","19,197.42","27,140.17","37,962.59","61,477.81","193,139.02"
uso_tarjeta_pct,"10,000.00",49.95,28.56,0.00,5.20,25.50,49.85,74.40,94.50,100.00
transacciones_mes,"10,000.00",17.98,4.30,4.00,11.00,15.00,18.00,21.00,25.00,36.00
gasto_mensual,"10,000.00","12,049.73","6,052.09",687.52,"4,118.49","7,579.33","11,115.26","15,384.69","23,483.93","56,173.87"
ecommerce_pct,"10,000.00",49.65,28.75,0.00,5.40,25.20,49.60,74.30,95.10,100.00
pagos_puntuales_pct,"10,000.00",85.00,8.65,70.00,71.60,77.50,85.00,92.50,98.50,100.00
app_logins_mes,"10,000.00",10.03,3.18,1.00,5.00,8.00,10.00,12.00,16.00,23.00
nps,"10,000.00",5.03,3.15,0.00,0.00,2.00,5.00,8.00,10.00,10.00


Registros fuera de rango: {'uso_tarjeta_pct': 0, 'ecommerce_pct': 0, 'pagos_puntuales_pct': 0, 'activacion_promos_pct': 0, 'abandono_riesgo': 0, 'nps': 0, 'satisfaccion': 0}
Campañas con conversión > apertura: 0


In [6]:
# ¿Las variables de clientes están relacionadas entre sí?
corr = clientes[NUM].corr()
fig = px.imshow(corr, color_continuous_scale=DIVERGENTE, zmin=-1, zmax=1, text_auto=".2f",
                aspect="auto", title="Correlación entre variables de clientes")
fig.update_layout(height=620, coloraxis_colorbar=dict(title="ρ"))
fig.show()
print("Máx |ρ| fuera de la diagonal:", round(corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max(), 3))

Máx |ρ| fuera de la diagonal: 0.028


**Lectura — calidad de datos**

- Base limpia: 10,000 clientes únicos, **0 nulos, 0 duplicados, 0 valores fuera de rango**; en campañas, ninguna tiene conversión > apertura.
- `ingreso_mensual` y `gasto_mensual` tienen cola derecha (máx. ~$193k y ~$56k): la mediana es mejor resumen que la media.
- ⚠️ **Hallazgo estructural:** la correlación máxima entre cualquier par de variables es |ρ| ≈ 0.03. Las variables de clientes son **independientes entre sí** (ingreso no se relaciona con gasto, uso no se relaciona con abandono, etc.). Es típico de un dataset dummy generado al azar y condiciona todo lo que sigue: no hay que esperar perfiles distintos por tarjeta o segmento.

## 3. Definición de particiones y mapeo a clientes

La partición de una campaña es `segmento_objetivo · producto`. Para llevarla a la base de clientes hay que
traducir el **producto de campaña** a las **tarjetas** de `tarjeta_santander`. Primero se valida si el segmento
que declara la campaña coincide con el segmento que realmente tienen esos tarjetahabientes.

In [7]:
display(pd.crosstab(clientes.tarjeta_santander, clientes.segmento, margins=True, margins_name="Total"))

segmento,Mass,Premier,Select,Total
tarjeta_santander,,,,
Aeromexico Blanca,1178,0,0,1178
Aeromexico Platinum,0,813,0,813
Fiesta Rewards Oro,686,0,0,686
Fiesta Rewards Platinum,0,520,0,520
Free,1998,0,0,1998
LikeU,2512,0,0,2512
Santander Gold,1000,0,0,1000
Santander Platinum,0,991,0,991
World Elite,0,0,302,302


In [8]:
# Producto de campaña -> tarjetas en la base de clientes
PRODUCTO_A_TARJETAS = {
    "LikeU": ["LikeU"],
    "Todas": sorted(clientes.tarjeta_santander.unique()),
    "Aeroméxico Platinum": ["Aeromexico Platinum"],
    "Fiesta Rewards": ["Fiesta Rewards Oro", "Fiesta Rewards Platinum"],
    "Platinum": ["Santander Platinum"],
    "World Elite": ["World Elite"],
}

campanas["particion_nombre"] = campanas.segmento_objetivo + " · " + campanas.producto
CODIGO = {
    "Mass · LikeU": "P1",
    "Todos · Todas": "P2",
    "Premier · Aeroméxico Platinum": "P3",
    "Select · Fiesta Rewards": "P4",
    "Select · Platinum": "P5",
    "Premier · World Elite": "P6",
}
assert set(CODIGO) == set(campanas.particion_nombre), "Hay particiones de campaña sin código asignado"
campanas["particion"] = campanas.particion_nombre.map(CODIGO)

PARTICIONES = {}
for nombre, cod in CODIGO.items():
    seg_obj, prod = nombre.split(" · ")
    tarjetas = PRODUCTO_A_TARJETAS[prod]
    mask = clientes.tarjeta_santander.isin(tarjetas)
    seg_real = sorted(clientes.loc[mask, "segmento"].unique())
    PARTICIONES[cod] = dict(nombre=nombre, segmento_obj=seg_obj, producto=prod, tarjetas=tarjetas, mask=mask,
                            consistente=(seg_obj == "Todos") or (seg_real == [seg_obj]),
                            segmento_real=", ".join(seg_real))

mapa = pd.DataFrame([
    {"particion": k, "nombre": v["nombre"], "tarjetas en base": ", ".join(v["tarjetas"]) if v["producto"] != "Todas" else "(todas)",
     "segmento real": v["segmento_real"], "segmento consistente": v["consistente"],
     "clientes": int(v["mask"].sum()), "% base": v["mask"].mean() * 100,
     "campañas": int((campanas.particion == k).sum())}
    for k, v in PARTICIONES.items()
]).set_index("particion")
display(mapa)

,nombre,tarjetas en base,segmento real,segmento consistente,clientes,% base,campañas
particion,,,,,,,
P1,Mass · LikeU,LikeU,Mass,True,2512,25.12,8
P2,Todos · Todas,(todas),"Mass, Premier, Select",True,10000,100.00,6
P3,Premier · Aeroméxico Platinum,Aeromexico Platinum,Premier,True,813,8.13,2
P4,Select · Fiesta Rewards,"Fiesta Rewards Oro, Fiesta Rewards Platinum","Mass, Premier",False,1206,12.06,1
P5,Select · Platinum,Santander Platinum,Premier,False,991,9.91,2
P6,Premier · World Elite,World Elite,Select,False,302,3.02,1


In [9]:
# Cobertura: ¿qué tarjetas solo reciben campañas masivas (P2)?
dirigidas = set(t for k, v in PARTICIONES.items() if k != "P2" for t in v["tarjetas"])
cobertura = (clientes.groupby("tarjeta_santander").size().rename("clientes").to_frame()
             .assign(campana_dirigida=lambda d: d.index.isin(dirigidas))
             .sort_values("clientes", ascending=False))
display(cobertura)
sin = cobertura.loc[~cobertura.campana_dirigida, "clientes"].sum()
print(f"Clientes sin ninguna campaña dirigida (solo P2 masivas): {sin:,} ({sin/len(clientes):.1%} de la base)")

,clientes,campana_dirigida
tarjeta_santander,,
LikeU,2512,True
Free,1998,False
Aeromexico Blanca,1178,False
Santander Gold,1000,False
Santander Platinum,991,True
Aeromexico Platinum,813,True
Fiesta Rewards Oro,686,True
Fiesta Rewards Platinum,520,True
World Elite,302,True


Clientes sin ninguna campaña dirigida (solo P2 masivas): 4,176 (41.8% de la base)


**Lectura — mapeo de particiones**

- **3 de 6 particiones son consistentes** con la base (P1 LikeU→Mass, P2 Todos, P3 Aeroméxico Platinum→Premier).
- ⚠️ **3 particiones tienen el segmento mal declarado** respecto a la base de clientes:
  - **P4 Select · Fiesta Rewards** → en la base Fiesta Rewards es **Mass (Oro) + Premier (Platinum)**.
  - **P5 Select · Platinum** → Santander Platinum es **Premier**.
  - **P6 Premier · World Elite** → World Elite es **Select**.
  
  Parece que Premier y Select están intercambiados en el catálogo de campañas (o en la base). Hay que confirmarlo con el dueño de los datos antes de usar `segmento` como filtro de audiencia; aquí se mapeó por **producto**, que no es ambiguo.
- **41.8% de la base (Free, Aeroméxico Blanca y Santander Gold) no tiene ninguna campaña dirigida**: solo la alcanzan las masivas de P2. Gold solo aparece de forma indirecta como *pool* de la campaña de upgrade (P5).

## 4. Panorama de campañas por partición

KPIs ponderados por `clientes_objetivo`. Se agregan dos métricas derivadas:
- **conversiones estimadas** = `clientes_objetivo × conversion_pct` (asumiendo que la conversión se mide sobre la base objetivo).
- **conversión / apertura** = qué tanto de lo que se abre termina en acción (eficiencia del mensaje).

In [10]:
KPI = ["apertura_pct", "conversion_pct", "incremento_gasto_pct", "incremento_uso_tarjeta_pct"]
KPI_LBL = {"apertura_pct": "Apertura %", "conversion_pct": "Conversión %",
           "incremento_gasto_pct": "Δ gasto %", "incremento_uso_tarjeta_pct": "Δ uso tarjeta %"}

campanas["conversiones_est"] = campanas.clientes_objetivo * campanas.conversion_pct / 100
campanas["conv_sobre_apertura"] = campanas.conversion_pct / campanas.apertura_pct * 100

def wavg(g, col, w="clientes_objetivo"):
    return np.average(g[col], weights=g[w])

def kpis_ponderados(df):
    return pd.Series({
        "campañas": len(df),
        "alcance": df.clientes_objetivo.sum(),
        "conversiones_est": df.conversiones_est.sum(),
        **{k: wavg(df, k) for k in KPI},
        "conv_sobre_apertura": df.conversiones_est.sum() / (df.clientes_objetivo * df.apertura_pct / 100).sum() * 100,
    })

resumen_camp = campanas.groupby("particion").apply(kpis_ponderados, include_groups=False)
resumen_camp.insert(0, "nombre", [PARTICIONES[p]["nombre"] for p in resumen_camp.index])
total = kpis_ponderados(campanas)
resumen_camp.loc["Total"] = ["Todas las campañas", *total.values]
display(resumen_camp)

,nombre,campañas,alcance,conversiones_est,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura
particion,,,,,,,,,
P1,Mass · LikeU,8.00,"1,350,000.00","462,050.00",77.90,34.23,14.19,21.29,43.93
P2,Todos · Todas,6.00,"2,470,000.00","981,400.00",85.77,39.73,20.72,26.16,46.33
P3,Premier · Aeroméxico Platinum,2.00,"87,000.00","27,450.00",75.55,31.55,17.03,19.52,41.76
P4,Select · Fiesta Rewards,1.00,"35,000.00","10,150.00",70.00,29.00,14.00,16.00,41.43
P5,Select · Platinum,2.00,"80,000.00","19,520.00",70.75,24.40,17.68,15.75,34.49
P6,Premier · World Elite,1.00,"12,000.00","2,640.00",71.00,22.00,24.00,12.00,30.99
Total,Todas las campañas,20.00,"4,034,000.00","1,503,210.00",82.44,37.26,18.35,24.05,45.20


In [11]:
r = resumen_camp.drop("Total").reset_index()
r["etq"] = r.particion + " · " + r.nombre
fig = make_subplots(rows=1, cols=4, shared_yaxes=True, subplot_titles=[KPI_LBL[k] for k in KPI],
                    horizontal_spacing=0.03)
for i, k in enumerate(KPI, start=1):
    fig.add_bar(y=r.etq, x=r[k], orientation="h", marker_color=PAL[0], showlegend=False,
                text=r[k].round(1), textposition="outside", cliponaxis=False,
                hovertemplate="%{y}<br>" + KPI_LBL[k] + ": %{x:.1f}<extra></extra>", row=1, col=i)
    ax = "" if i == 1 else str(i)
    fig.add_shape(type="line", x0=total[k], x1=total[k], xref=f"x{ax}", y0=0, y1=1, yref=f"y{ax} domain",
                  line=dict(color="#52514e", dash="dot", width=1.5), layer="above")
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=380, title="KPIs ponderados por partición (línea punteada = promedio total ponderado)")
fig.show()

In [12]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=["Alcance (clientes objetivo)", "Conversiones estimadas"])
for i, col in enumerate(["alcance", "conversiones_est"], start=1):
    fig.add_bar(y=r.etq, x=r[col], orientation="h", marker_color=PAL[0], showlegend=False,
                text=(r[col] / 1e3).round(0).astype(int).astype(str) + "k", textposition="outside", cliponaxis=False,
                hovertemplate="%{y}<br>%{x:,.0f}<extra></extra>", row=1, col=i)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=340, title="Escala de cada partición")
fig.show()

In [13]:
# Cada campaña: apertura vs conversión (etiqueta = partición)
c = campanas.copy()
fig = px.scatter(c, x="apertura_pct", y="conversion_pct", size="clientes_objetivo", size_max=40, text="particion",
                 hover_data={"campaign_id": True, "campana": True, "particion_nombre": True, "anio": True,
                             "clientes_objetivo": ":,", "particion": False},
                 title="Apertura vs conversión por campaña (tamaño = clientes objetivo)")
fig.update_traces(marker=dict(color=PAL[0], opacity=0.55, line=dict(color="white", width=2)),
                  textposition="middle right", textfont=dict(color="#52514e"))
fig.update_layout(height=480, xaxis_title="Apertura %", yaxis_title="Conversión %")
fig.show()
print("Correlación apertura–conversión entre campañas:", round(c.apertura_pct.corr(c.conversion_pct), 2))

Correlación apertura–conversión entre campañas: 0.94


In [14]:
# Evolución 2024 -> 2025 dentro de cada partición
anual = (campanas.groupby(["particion", "anio"]).apply(kpis_ponderados, include_groups=False)[KPI + ["campañas"]]
         .reset_index())
display(anual.pivot(index="particion", columns="anio", values=["campañas", "conversion_pct", "incremento_gasto_pct"]))

fig = make_subplots(rows=1, cols=4, subplot_titles=[KPI_LBL[k] for k in KPI], horizontal_spacing=0.05)
for j, anio in enumerate([2024, 2025]):
    d = anual[anual.anio == anio]
    for i, k in enumerate(KPI, start=1):
        fig.add_bar(x=d.particion, y=d[k], name=str(anio), marker_color=PAL[j], legendgroup=str(anio),
                    showlegend=(i == 1), hovertemplate="%{x} " + str(anio) + "<br>%{y:.1f}<extra></extra>",
                    row=1, col=i)
fig.update_layout(height=390, barmode="group", title="KPIs ponderados por año y partición",
                  margin=dict(t=100), legend=dict(orientation="h", y=1.3, x=1, xanchor="right"))
fig.show()

campañas      conversion_pct       incremento_gasto_pct      
anio          2024 2025           2024  2025                 2024  2025
particion                                                              
P1            4.00 4.00          26.76 39.28                11.28 16.15
P2            3.00 3.00          38.11 41.86                20.36 21.21
P3            1.00 1.00          33.00 30.00                18.00 16.00
P4            1.00  NaN          29.00   NaN                14.00   NaN
P5            1.00 1.00          12.00 28.00                20.00 17.00
P6             NaN 1.00            NaN 22.00                  NaN 24.00

**Lectura — panorama de campañas**

- **P2 (masivas)** domina en escala (2.47M de alcance, ~65% de las conversiones estimadas) y en desempeño: mejor apertura (85.8%), conversión (39.7%), Δ gasto (20.7%) y Δ uso (26.2%).
- **P1 (LikeU)** es la segunda en volumen (8 campañas, 1.35M), con conversión sobre el promedio de las dirigidas pero el **Δ gasto más bajo (14.2%)**: el cashback por categoría mueve frecuencia más que ticket.
- Las particiones premium (**P5, P6**) convierten poco (24% y 22%) y tienen la peor eficiencia post-apertura (~31–34%), pero **P6 World Elite logra el mayor Δ gasto (24%)**: poca gente responde, pero quien responde gasta mucho más.
- Apertura y conversión están muy correlacionadas entre campañas (ρ = 0.94): el gancho del mensaje explica buena parte del resultado.
- **2024 → 2025:** P1 mejora fuerte (conversión 26.8% → 39.3%, Δ gasto 11.3% → 16.2%) al pasar de cashback genérico a cashback con % explícito (6%, 5%, Baby). P5 también mejora al cambiar *upgrade* (12%) por *MSI Viajes* (28%). P3 se desgasta ligeramente.
- Casos aislados débiles: **CAM019 Reactivación** (17% conversión, 59% apertura) y **CAM009 Upgrade Gold→Platinum** (12%).

## 5. EDA por partición

Funciones reutilizables para que cada partición se analice con el mismo lente:

- **`tabla_campanas`** — campañas de la partición y sus KPIs.
- **`perfil_vs_resto`** — medias de la partición vs el resto de la base, diferencia % y **d de Cohen**
  (|d| < 0.2 = diferencia despreciable, 0.2–0.5 pequeña, 0.5–0.8 mediana, > 0.8 grande).
- **`distribuciones`** — boxplots de las métricas palanca, partición vs resto.
- **`oportunidades`** — bolsas de clientes accionables (% de la partición vs % de la base).
- **`impacto_estimado`** — escenario ilustrativo: si se aplican los KPIs de las campañas de la partición a su población.

In [15]:
VARS_PERFIL = ["edad", "antiguedad_meses", "ingreso_mensual", "uso_tarjeta_pct", "transacciones_mes",
               "gasto_mensual", "ecommerce_pct", "pagos_puntuales_pct", "app_logins_mes", "nps",
               "satisfaccion", "abandono_riesgo", "activacion_promos_pct"]

FLAGS = {
    "Bajo uso de tarjeta (<25%)": lambda d: d.uso_tarjeta_pct < 25,
    "Baja activación de promos (<25%)": lambda d: d.activacion_promos_pct < 25,
    "Alto riesgo de abandono (≥0.75)": lambda d: d.abandono_riesgo >= 0.75,
    "Detractores NPS (0–6)": lambda d: d.nps <= 6,
    "Baja adopción digital (≤6 logins/mes)": lambda d: d.app_logins_mes <= 6,
    "Alto gasto (top 25% base)": lambda d: d.gasto_mensual >= clientes.gasto_mensual.quantile(.75),
    "Prioridad retención: alto riesgo + baja activación": lambda d: (d.abandono_riesgo >= 0.75) & (d.activacion_promos_pct < 25),
}

def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp

def magnitud(d):
    d = abs(d)
    return "despreciable" if d < .2 else "pequeña" if d < .5 else "mediana" if d < .8 else "grande"

def tabla_campanas(p):
    cols = ["campaign_id", "anio", "campana", "clientes_objetivo", *KPI, "conv_sobre_apertura"]
    t = campanas.loc[campanas.particion == p, cols].sort_values(["anio", "campaign_id"])
    display(t.set_index("campaign_id"))
    if len(t) > 1:
        fig = go.Figure()
        for i, k in enumerate(KPI):
            fig.add_bar(y=t.campaign_id + " · " + t.campana, x=t[k], name=KPI_LBL[k], orientation="h",
                        marker_color=PAL[i], hovertemplate="%{y}<br>" + KPI_LBL[k] + ": %{x}<extra></extra>")
        fig.update_layout(barmode="group", height=110 + 70 * len(t), yaxis_autorange="reversed",
                          title=f"{p} · KPIs por campaña", legend=dict(orientation="h", y=1.08, x=1, xanchor="right"),
                          bargroupgap=0.08)
        fig.show()
    return t

def perfil_vs_resto(p, mask=None, etiqueta=None):
    mask = PARTICIONES[p]["mask"] if mask is None else mask
    a, b = clientes[mask], clientes[~mask]
    t = pd.DataFrame({
        "media partición": a[VARS_PERFIL].mean(),
        "media resto": b[VARS_PERFIL].mean(),
        "mediana partición": a[VARS_PERFIL].median(),
    })
    t["dif %"] = (t["media partición"] / t["media resto"] - 1) * 100
    t["d Cohen"] = [cohen_d(a[v], b[v]) for v in VARS_PERFIL]
    t["magnitud"] = t["d Cohen"].map(magnitud)
    display(t)
    s = t["d Cohen"].sort_values()
    fig = go.Figure(go.Bar(y=s.index, x=s.values, orientation="h",
                           marker_color=np.where(s.values >= 0, POS, NEG),
                           hovertemplate="%{y}<br>d = %{x:.3f}<extra></extra>"))
    fig.add_vrect(x0=-0.2, x1=0.2, fillcolor=NEUTRO, opacity=0.8, line_width=0, layer="below",
                  annotation_text="zona despreciable (|d|<0.2)", annotation_position="top left")
    fig.update_layout(height=430, xaxis=dict(range=[-0.5, 0.5], title="d de Cohen (partición vs resto)"),
                      title=f"{etiqueta or p} · ¿en qué se diferencia del resto de la base?")
    fig.show()
    return t

def distribuciones(p, variables, mask=None):
    mask = PARTICIONES[p]["mask"] if mask is None else mask
    g = np.where(mask, p, "Resto de la base")
    fig = make_subplots(rows=1, cols=len(variables), subplot_titles=variables, horizontal_spacing=0.06)
    for i, v in enumerate(variables, start=1):
        for j, (grupo, color) in enumerate([(p, PAL[0]), ("Resto de la base", GRIS)]):
            fig.add_box(y=clientes.loc[g == grupo, v], name=grupo, marker_color=color, boxmean=True,
                        legendgroup=grupo, showlegend=(i == 1), row=1, col=i)
    fig.update_xaxes(showticklabels=False)
    fig.update_layout(height=380, title=f"{p} · distribución de métricas palanca (◇ = media)",
                      legend=dict(orientation="h", y=1.18, x=1, xanchor="right"))
    fig.show()

def oportunidades(p, mask=None):
    mask = PARTICIONES[p]["mask"] if mask is None else mask
    a = clientes[mask]
    t = pd.DataFrame({
        "clientes": {k: int(f(a).sum()) for k, f in FLAGS.items()},
        "% partición": {k: f(a).mean() * 100 for k, f in FLAGS.items()},
        "% base": {k: f(clientes).mean() * 100 for k, f in FLAGS.items()},
    })
    t["dif pp"] = t["% partición"] - t["% base"]
    display(t)
    return t

def impacto_estimado(p, mask=None):
    mask = PARTICIONES[p]["mask"] if mask is None else mask
    a = clientes[mask]
    k = resumen_camp.loc[p]
    conv = len(a) * k.conversion_pct / 100
    gasto_base = a.gasto_mensual.mean()
    inc = conv * gasto_base * k.incremento_gasto_pct / 100
    display(Markdown(
        f"**Escenario ilustrativo {p}** (muestra de {len(a):,} clientes, KPIs ponderados de sus campañas): "
        f"~{conv:,.0f} conversiones ({k.conversion_pct:.1f}%), gasto medio actual ${gasto_base:,.0f}/mes → "
        f"incremento estimado de **${inc:,.0f}/mes** en gasto (+{k.incremento_gasto_pct:.1f}% en convertidos)."))
    return dict(particion=p, clientes=len(a), conversiones=conv, gasto_medio=gasto_base, incremento_gasto_mes=inc)

IMPACTOS = {}

### 5.1 P1 · Mass · LikeU

Partición más activa del portafolio (8 campañas, en su mayoría **cashback por categoría**: farmacias,
restaurantes, telecom, contactless, referidos). Mapeo directo: tarjeta **LikeU** (100% Mass en la base, consistente).
Palancas naturales: activación de promociones, uso de tarjeta, e-commerce y frecuencia de transacciones.

#### P1 · a) Campañas

In [16]:
camp_P1 = tabla_campanas("P1")

,anio,campana,clientes_objetivo,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura
campaign_id,,,,,,,,
CAM001,2024,Cashback LikeU Farmacias,150000,72,28,12,18,38.89
CAM002,2024,Cashback LikeU Restaurantes,180000,75,31,15,22,41.33
CAM003,2024,Cashback LikeU Telecom,120000,68,25,9,14,36.76
CAM010,2024,Referidos LikeU,95000,67,19,6,10,28.36
CAM011,2025,Cashback Baby LikeU,350000,83,41,19,28,49.40
CAM015,2025,Promoción Contactless,160000,79,34,10,15,43.04
CAM017,2025,Cashback Farmacias 6%,140000,82,38,14,23,46.34
CAM018,2025,Cashback Restaurantes 5%,155000,85,42,18,26,49.41


#### P1 · b) Población

In [17]:
m = PARTICIONES["P1"]["mask"]
print(f"Clientes en P1: {m.sum():,} ({m.mean():.1%} de la base)")
display(clientes[m].groupby(["tarjeta_santander", "segmento"]).size().rename("clientes").to_frame())

Clientes en P1: 2,512 (25.1% de la base)


,,clientes
tarjeta_santander,segmento,
LikeU,Mass,2512


#### P1 · c) Perfil vs resto de la base

In [18]:
perfil_P1 = perfil_vs_resto("P1", etiqueta="P1 · Mass · LikeU")

,media partición,media resto,mediana partición,dif %,d Cohen,magnitud
edad,46.18,46.58,46.00,-0.87,-0.02,despreciable
antiguedad_meses,89.97,92.22,89.00,-2.44,-0.04,despreciable
ingreso_mensual,"30,486.61","30,674.71","27,314.34",-0.61,-0.01,despreciable
uso_tarjeta_pct,50.57,49.74,50.60,1.67,0.03,despreciable
transacciones_mes,17.95,18.00,18.00,-0.29,-0.01,despreciable
gasto_mensual,"12,086.01","12,037.56","11,293.31",0.40,0.01,despreciable
ecommerce_pct,50.17,49.47,49.95,1.40,0.02,despreciable
pagos_puntuales_pct,85.01,85.00,84.90,0.01,0.00,despreciable
app_logins_mes,10.06,10.02,10.00,0.38,0.01,despreciable
nps,5.07,5.02,5.00,0.97,0.02,despreciable


#### P1 · d) Distribuciones de métricas palanca

In [19]:
distribuciones("P1", ['activacion_promos_pct', 'uso_tarjeta_pct', 'ecommerce_pct', 'transacciones_mes', 'gasto_mensual'])

#### P1 · e) Oportunidades e impacto

In [20]:
op_P1 = oportunidades("P1")
IMPACTOS["P1"] = impacto_estimado("P1")

,clientes,% partición,% base,dif pp
Bajo uso de tarjeta (<25%),577,22.97,24.37,-1.40
Baja activación de promos (<25%),628,25.00,24.49,0.51
Alto riesgo de abandono (≥0.75),614,24.44,24.88,-0.44
Detractores NPS (0–6),1589,63.26,63.46,-0.20
Baja adopción digital (≤6 logins/mes),305,12.14,12.87,-0.73
Alto gasto (top 25% base),642,25.56,25.00,0.56
Prioridad retención: alto riesgo + baja activación,142,5.65,6.27,-0.62


**Escenario ilustrativo P1** (muestra de 2,512 clientes, KPIs ponderados de sus campañas): ~860 conversiones (34.2%), gasto medio actual $12,086/mes → incremento estimado de **$1,473,984/mes** en gasto (+14.2% en convertidos).

**Lectura — P1 Mass · LikeU**

- 2,512 clientes (25% de la base), la población dirigida más grande.
- **Perfil indistinguible del resto de la base**: todas las d de Cohen son |d| < 0.05 (despreciables). LikeU no es más digital ni más activo en promociones que el resto, a pesar de que el glosario la describe como tarjeta de "alta interacción con promociones".
- Campañas: las 2025 con **% de cashback explícito** (CAM017, CAM018, CAM011) superan claramente a las 2024 genéricas; **Referidos (CAM010)** es la más débil en todos los KPIs.
- Oportunidad: ~628 clientes (25%) con activación de promos < 25% y 142 en prioridad de retención (alto riesgo + baja activación). Escenario ilustrativo: ~860 conversiones y +$1.47M/mes de gasto en la muestra.

### 5.2 P2 · Todos · Todas

Campañas masivas a toda la base (Hot Sale, Buen Fin, activación digital/app, reactivación de inactivas).
La población es **toda la base**, así que la comparación "vs resto" no aplica: aquí el análisis se hace
**por tarjeta** dentro de la partición, y se dimensionan las bolsas específicas de cada campaña
(inactivos para CAM019, baja adopción digital para CAM004/CAM014).

#### P2 · a) Campañas

In [21]:
camp_P2 = tabla_campanas("P2")

,anio,campana,clientes_objetivo,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura
campaign_id,,,,,,,,
CAM004,2024,Activación Tarjeta Digital,250000,81,42,7,11,51.85
CAM005,2024,Hot Sale MSI,500000,84,35,21,27,41.67
CAM006,2024,Buen Fin Cashback,650000,88,39,25,31,44.32
CAM013,2025,Hot Sale Cashback 15%,700000,90,43,29,34,47.78
CAM014,2025,Activación App Santander,280000,86,47,5,9,54.65
CAM019,2025,Reactivación Tarjetas Inactivas,90000,59,17,11,21,28.81


#### P2 · b) Población y perfil por tarjeta

In [22]:
por_tarjeta = clientes.groupby("tarjeta_santander")[VARS_PERFIL].mean()
z = (por_tarjeta - clientes[VARS_PERFIL].mean()) / clientes[VARS_PERFIL].std()
display(por_tarjeta.assign(clientes=clientes.tarjeta_santander.value_counts()))
fig = px.imshow(z.T, color_continuous_scale=DIVERGENTE, zmin=-0.3, zmax=0.3, text_auto=".2f", aspect="auto",
                title="P2 · desviación de cada tarjeta vs la media de la base (en desviaciones estándar)")
fig.update_layout(height=560, coloraxis_colorbar=dict(title="z"))
fig.show()
print("Máxima desviación |z| de cualquier tarjeta en cualquier variable:", round(z.abs().max().max(), 3))

,edad,antiguedad_meses,ingreso_mensual,uso_tarjeta_pct,transacciones_mes,gasto_mensual,ecommerce_pct,pagos_puntuales_pct,app_logins_mes,nps,satisfaccion,abandono_riesgo,activacion_promos_pct,clientes
tarjeta_santander,,,,,,,,,,,,,,
Aeromexico Blanca,46.57,89.42,"31,167.85",47.86,17.92,"12,160.67",47.84,84.91,10.08,5.05,3.00,0.50,50.32,1178
Aeromexico Platinum,46.38,92.85,"30,515.41",50.82,18.12,"11,949.03",49.07,85.00,10.00,5.09,2.96,0.50,50.16,813
Fiesta Rewards Oro,46.50,94.53,"31,151.50",49.65,17.97,"12,051.12",49.38,84.14,10.13,5.09,2.92,0.50,53.09,686
Fiesta Rewards Platinum,46.29,89.08,"30,574.76",50.93,17.51,"11,796.13",50.49,84.76,10.03,4.81,3.05,0.48,49.27,520
Free,46.54,92.06,"30,999.49",50.36,18.11,"11,983.37",50.28,85.12,10.02,4.98,2.98,0.50,50.37,1998
LikeU,46.18,89.97,"30,486.61",50.57,17.95,"12,086.01",50.17,85.01,10.06,5.07,2.96,0.50,50.52,2512
Santander Gold,46.84,93.57,"29,584.71",49.25,18.05,"12,195.57",49.08,85.46,10.04,4.97,2.96,0.50,50.23,1000
Santander Platinum,46.67,93.72,"30,446.16",49.08,17.95,"12,136.82",50.19,85.06,9.94,5.13,3.05,0.51,49.32,991
World Elite,46.98,93.30,"30,479.53",52.02,18.15,"11,690.21",48.98,85.19,9.83,4.96,2.95,0.48,51.76,302


Máxima desviación |z| de cualquier tarjeta en cualquier variable: 0.111


#### P2 · c) Distribuciones por tarjeta

In [23]:
orden_t = clientes.tarjeta_santander.value_counts().index
fig = make_subplots(rows=1, cols=3, subplot_titles=["uso_tarjeta_pct", "app_logins_mes", "gasto_mensual"],
                    shared_yaxes=True, horizontal_spacing=0.03)
for i, v in enumerate(["uso_tarjeta_pct", "app_logins_mes", "gasto_mensual"], start=1):
    for t in orden_t:
        fig.add_box(x=clientes.loc[clientes.tarjeta_santander == t, v], name=t, orientation="h",
                    marker_color=PAL[0], showlegend=False, boxmean=True, row=1, col=i)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=480, title="P2 · métricas palanca por tarjeta (◇ = media)")
fig.show()

#### P2 · d) Bolsas por campaña masiva

In [24]:
BOLSAS_P2 = {
    "CAM019 Reactivación: inactivos (uso <10%)": clientes.uso_tarjeta_pct < 10,
    "CAM004/CAM014 Activación digital: ≤6 logins/mes": clientes.app_logins_mes <= 6,
    "CAM005/006/013 Hot Sale/Buen Fin: alta activación promos (≥75%)": clientes.activacion_promos_pct >= 75,
    "Retención: alto riesgo (≥0.75) y bajo uso (<25%)": (clientes.abandono_riesgo >= .75) & (clientes.uso_tarjeta_pct < 25),
}
print("Tamaño de cada bolsa en la base:")
display(pd.DataFrame({"clientes": {k: int(m.sum()) for k, m in BOLSAS_P2.items()},
                      "% base": {k: m.mean() * 100 for k, m in BOLSAS_P2.items()}}))
print("% de clientes de cada tarjeta que cae en la bolsa:")
bolsas = pd.DataFrame({k: m.groupby(clientes.tarjeta_santander).mean() * 100 for k, m in BOLSAS_P2.items()})
display(bolsas.round(1))

Tamaño de cada bolsa en la base:


,clientes,% base
CAM019 Reactivación: inactivos (uso <10%),944,9.44
CAM004/CAM014 Activación digital: ≤6 logins/mes,1287,12.87
CAM005/006/013 Hot Sale/Buen Fin: alta activación promos (≥75%),2546,25.46
Retención: alto riesgo (≥0.75) y bajo uso (<25%),606,6.06


% de clientes de cada tarjeta que cae en la bolsa:


,CAM019 Reactivación: inactivos (uso <10%),CAM004/CAM014 Activación digital: ≤6 logins/mes,CAM005/006/013 Hot Sale/Buen Fin: alta activación promos (≥75%),Retención: alto riesgo (≥0.75) y bajo uso (<25%)
tarjeta_santander,,,,
Aeromexico Blanca,9.90,10.80,24.90,7.50
Aeromexico Platinum,8.50,13.50,25.00,6.40
Fiesta Rewards Oro,9.60,12.50,28.60,5.00
Fiesta Rewards Platinum,10.20,14.00,22.10,4.60
Free,9.60,13.10,26.50,6.30
LikeU,8.70,12.10,25.40,5.40
Santander Gold,11.10,13.90,24.60,6.40
Santander Platinum,9.60,13.90,24.40,6.70
World Elite,7.90,15.60,27.80,5.60


#### P2 · e) Oportunidades e impacto

In [25]:
op_P2 = oportunidades("P2")
IMPACTOS["P2"] = impacto_estimado("P2")

,clientes,% partición,% base,dif pp
Bajo uso de tarjeta (<25%),2437,24.37,24.37,0.00
Baja activación de promos (<25%),2449,24.49,24.49,0.00
Alto riesgo de abandono (≥0.75),2488,24.88,24.88,0.00
Detractores NPS (0–6),6346,63.46,63.46,0.00
Baja adopción digital (≤6 logins/mes),1287,12.87,12.87,0.00
Alto gasto (top 25% base),2500,25.00,25.00,0.00
Prioridad retención: alto riesgo + baja activación,627,6.27,6.27,0.00


**Escenario ilustrativo P2** (muestra de 10,000 clientes, KPIs ponderados de sus campañas): ~3,973 conversiones (39.7%), gasto medio actual $12,050/mes → incremento estimado de **$9,922,353/mes** en gasto (+20.7% en convertidos).

**Lectura — P2 Todos · Todas**

- Al ser toda la base, la pregunta relevante es si alguna tarjeta se comporta distinto: **no**. La desviación máxima de cualquier tarjeta en cualquier variable es |z| ≈ 0.11 (Fiesta Rewards Oro en activación de promos, 53% vs 50%).
- Bolsas por campaña masiva (en la muestra): **944 inactivos** (uso < 10%) para Reactivación, **1,287 con baja adopción digital** para Activación App/Digital, **2,546 muy activos en promos** para Hot Sale/Buen Fin y **606 en riesgo** (alto abandono + bajo uso). La proporción es casi igual en todas las tarjetas.
- CAM019 Reactivación es la peor masiva (17%) pese a que su bolsa existe: el problema es el mensaje/incentivo, no el tamaño de la audiencia.
- Las campañas de **activación (CAM004, CAM014)** tienen la mejor conversión sobre apertura (52–55%) pero el menor Δ gasto (5–7%): sirven para adopción, no para monetizar.

### 5.3 P3 · Premier · Aeroméxico Platinum

Campañas de recompensas de viaje (CAM007 Rewards Viajes 2024, CAM012 Compra y Gana 2025).
Mapeo directo: **Aeromexico Platinum** (100% Premier en la base, consistente). Palancas: gasto, uso y
e-commerce (compra de vuelos), riesgo de abandono en un segmento de alto valor.

#### P3 · a) Campañas

In [26]:
camp_P3 = tabla_campanas("P3")

,anio,campana,clientes_objetivo,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura
campaign_id,,,,,,,,
CAM007,2024,Aeroméxico Rewards Viajes,45000,77,33,18,20,42.86
CAM012,2025,Compra y Gana Aeroméxico,42000,74,30,16,19,40.54


#### P3 · b) Población

In [27]:
m = PARTICIONES["P3"]["mask"]
print(f"Clientes en P3: {m.sum():,} ({m.mean():.1%} de la base)")
display(clientes[m].groupby(["tarjeta_santander", "segmento"]).size().rename("clientes").to_frame())

Clientes en P3: 813 (8.1% de la base)


,,clientes
tarjeta_santander,segmento,
Aeromexico Platinum,Premier,813


#### P3 · c) Perfil vs resto de la base

In [28]:
perfil_P3 = perfil_vs_resto("P3", etiqueta="P3 · Premier · Aeroméxico Platinum")

,media partición,media resto,mediana partición,dif %,d Cohen,magnitud
edad,46.38,46.49,46.00,-0.24,-0.01,despreciable
antiguedad_meses,92.85,91.55,94.00,1.42,0.03,despreciable
ingreso_mensual,"30,515.41","30,637.37","27,108.87",-0.40,-0.01,despreciable
uso_tarjeta_pct,50.82,49.87,50.90,1.91,0.03,despreciable
transacciones_mes,18.12,17.97,18.00,0.81,0.03,despreciable
gasto_mensual,"11,949.03","12,058.64","11,004.43",-0.91,-0.02,despreciable
ecommerce_pct,49.07,49.70,48.30,-1.27,-0.02,despreciable
pagos_puntuales_pct,85.00,85.00,84.60,0.00,0.00,despreciable
app_logins_mes,10.00,10.03,10.00,-0.34,-0.01,despreciable
nps,5.09,5.03,5.00,1.21,0.02,despreciable


#### P3 · d) Distribuciones de métricas palanca

In [29]:
distribuciones("P3", ['gasto_mensual', 'uso_tarjeta_pct', 'ecommerce_pct', 'ingreso_mensual', 'abandono_riesgo'])

#### P3 · e) Oportunidades e impacto

In [30]:
op_P3 = oportunidades("P3")
IMPACTOS["P3"] = impacto_estimado("P3")

,clientes,% partición,% base,dif pp
Bajo uso de tarjeta (<25%),201,24.72,24.37,0.35
Baja activación de promos (<25%),201,24.72,24.49,0.23
Alto riesgo de abandono (≥0.75),202,24.85,24.88,-0.03
Detractores NPS (0–6),513,63.10,63.46,-0.36
Baja adopción digital (≤6 logins/mes),110,13.53,12.87,0.66
Alto gasto (top 25% base),194,23.86,25.00,-1.14
Prioridad retención: alto riesgo + baja activación,54,6.64,6.27,0.37


**Escenario ilustrativo P3** (muestra de 813 clientes, KPIs ponderados de sus campañas): ~257 conversiones (31.6%), gasto medio actual $11,949/mes → incremento estimado de **$522,126/mes** en gasto (+17.0% en convertidos).

**Lectura — P3 Premier · Aeroméxico Platinum**

- 813 clientes (8%), segmento consistente.
- Perfil sin diferencias relevantes vs el resto (|d| ≤ 0.03); incluso el gasto y el e-commerce están en el promedio, cuando se esperaría un perfil viajero de alto gasto.
- KPIs sólidos y estables (conversión ~31.6%, Δ gasto ~17%), con leve caída de 2024 a 2025 (CAM012 < CAM007): posible fatiga de la mecánica.
- Oportunidad: 194 clientes de alto gasto (top 25% de la base) y 54 en prioridad de retención. Escenario: ~257 conversiones, +$522k/mes.

### 5.4 P4 · Select · Fiesta Rewards

Una campaña (CAM008 Fiesta Rewards Hoteles, 2024). ⚠️ **Inconsistencia de segmento**: la campaña declara
Select, pero en la base **Fiesta Rewards Oro es Mass y Fiesta Rewards Platinum es Premier** (no hay Select).
Se mapea por producto (ambas tarjetas) y se compara Oro vs Platinum dentro de la partición.

#### P4 · a) Campañas

In [31]:
camp_P4 = tabla_campanas("P4")

,anio,campana,clientes_objetivo,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura
campaign_id,,,,,,,,
CAM008,2024,Fiesta Rewards Hoteles,35000,70,29,14,16,41.43


#### P4 · b) Población

In [32]:
m = PARTICIONES["P4"]["mask"]
print(f"Clientes en P4: {m.sum():,} ({m.mean():.1%} de la base)")
display(clientes[m].groupby(["tarjeta_santander", "segmento"]).size().rename("clientes").to_frame())

Clientes en P4: 1,206 (12.1% de la base)


,,clientes
tarjeta_santander,segmento,
Fiesta Rewards Oro,Mass,686
Fiesta Rewards Platinum,Premier,520


#### P4 · c) Perfil vs resto de la base

In [33]:
perfil_P4 = perfil_vs_resto("P4", etiqueta="P4 · Select · Fiesta Rewards")

,media partición,media resto,mediana partición,dif %,d Cohen,magnitud
edad,46.41,46.49,47.00,-0.17,-0.00,despreciable
antiguedad_meses,92.18,91.58,92.00,0.65,0.01,despreciable
ingreso_mensual,"30,902.82","30,589.69","27,487.78",1.02,0.02,despreciable
uso_tarjeta_pct,50.20,49.92,50.85,0.58,0.01,despreciable
transacciones_mes,17.77,18.01,18.00,-1.37,-0.06,despreciable
gasto_mensual,"11,941.17","12,064.62","10,771.12",-1.02,-0.02,despreciable
ecommerce_pct,49.86,49.62,49.75,0.48,0.01,despreciable
pagos_puntuales_pct,84.41,85.08,84.50,-0.79,-0.08,despreciable
app_logins_mes,10.09,10.02,10.00,0.63,0.02,despreciable
nps,4.97,5.04,5.00,-1.44,-0.02,despreciable


#### P4 · d) Distribuciones de métricas palanca

In [34]:
distribuciones("P4", ['gasto_mensual', 'activacion_promos_pct', 'uso_tarjeta_pct', 'ingreso_mensual', 'abandono_riesgo'])

#### P4 · d.2) Oro vs Platinum dentro de Fiesta Rewards

In [35]:
fr = clientes[PARTICIONES["P4"]["mask"]]
oro = fr.tarjeta_santander == "Fiesta Rewards Oro"
comp = pd.DataFrame({"Oro (Mass)": fr[oro][VARS_PERFIL].mean(), "Platinum (Premier)": fr[~oro][VARS_PERFIL].mean()})
comp["d Cohen (Oro vs Plat)"] = [cohen_d(fr[oro][v], fr[~oro][v]) for v in VARS_PERFIL]
display(comp)

,Oro (Mass),Platinum (Premier),d Cohen (Oro vs Plat)
edad,46.50,46.29,0.01
antiguedad_meses,94.53,89.08,0.11
ingreso_mensual,"31,151.50","30,574.76",0.04
uso_tarjeta_pct,49.65,50.93,-0.04
transacciones_mes,17.97,17.51,0.11
gasto_mensual,"12,051.12","11,796.13",0.04
ecommerce_pct,49.38,50.49,-0.04
pagos_puntuales_pct,84.14,84.76,-0.07
app_logins_mes,10.13,10.03,0.03
nps,5.09,4.81,0.09


#### P4 · e) Oportunidades e impacto

In [36]:
op_P4 = oportunidades("P4")
IMPACTOS["P4"] = impacto_estimado("P4")

,clientes,% partición,% base,dif pp
Bajo uso de tarjeta (<25%),287,23.80,24.37,-0.57
Baja activación de promos (<25%),289,23.96,24.49,-0.53
Alto riesgo de abandono (≥0.75),283,23.47,24.88,-1.41
Detractores NPS (0–6),768,63.68,63.46,0.22
Baja adopción digital (≤6 logins/mes),159,13.18,12.87,0.31
Alto gasto (top 25% base),296,24.54,25.00,-0.46
Prioridad retención: alto riesgo + baja activación,60,4.98,6.27,-1.29


**Escenario ilustrativo P4** (muestra de 1,206 clientes, KPIs ponderados de sus campañas): ~350 conversiones (29.0%), gasto medio actual $11,941/mes → incremento estimado de **$584,683/mes** en gasto (+14.0% en convertidos).

**Lectura — P4 Select · Fiesta Rewards**

- 1,206 clientes (686 Oro/Mass + 520 Platinum/Premier). ⚠️ Ninguno es Select: si la campaña se ejecutara filtrando por segmento Select, **no alcanzaría a ningún tarjetahabiente Fiesta Rewards**.
- Perfil vs resto: todo despreciable (|d| ≤ 0.08). Oro vs Platinum tampoco se diferencian (|d| ≤ 0.13): Oro tiene ligeramente más activación de promos (53% vs 49%) y más antigüedad.
- Una sola campaña (CAM008, 2024; 29% conversión, Δ gasto 14%) y **no se repitió en 2025**: es candidata a relanzarse con la mecánica de % explícito que funcionó en P1.
- Escenario: ~350 conversiones, +$585k/mes.

### 5.5 P5 · Select · Platinum

Dos campañas: CAM009 *Upgrade Gold a Platinum* (2024) y CAM016 *MSI Viajes Verano* (2025).
⚠️ **Inconsistencia de segmento**: la campaña declara Select, pero **Santander Platinum es Premier** en la base.
Además, CAM009 apunta en realidad a clientes **Gold** (candidatos a upgrade), por lo que se analizan dos
poblaciones: tarjetahabientes Platinum y el *pool* de upgrade Gold.

#### P5 · a) Campañas

In [37]:
camp_P5 = tabla_campanas("P5")

,anio,campana,clientes_objetivo,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura
campaign_id,,,,,,,,
CAM009,2024,Upgrade Gold a Platinum,18000,63,12,20,8,19.05
CAM016,2025,MSI Viajes Verano,62000,73,28,17,18,38.36


#### P5 · b) Población

In [38]:
m = PARTICIONES["P5"]["mask"]
print(f"Clientes en P5: {m.sum():,} ({m.mean():.1%} de la base)")
display(clientes[m].groupby(["tarjeta_santander", "segmento"]).size().rename("clientes").to_frame())

Clientes en P5: 991 (9.9% de la base)


,,clientes
tarjeta_santander,segmento,
Santander Platinum,Premier,991


#### P5 · c) Perfil vs resto de la base

In [39]:
perfil_P5 = perfil_vs_resto("P5", etiqueta="P5 · Select · Platinum")

,media partición,media resto,mediana partición,dif %,d Cohen,magnitud
edad,46.67,46.46,47.00,0.46,0.01,despreciable
antiguedad_meses,93.72,91.43,97.00,2.50,0.04,despreciable
ingreso_mensual,"30,446.16","30,647.40","26,936.84",-0.66,-0.01,despreciable
uso_tarjeta_pct,49.08,50.05,49.70,-1.93,-0.03,despreciable
transacciones_mes,17.95,17.99,18.00,-0.22,-0.01,despreciable
gasto_mensual,"12,136.82","12,040.15","11,292.80",0.80,0.02,despreciable
ecommerce_pct,50.19,49.59,51.50,1.21,0.02,despreciable
pagos_puntuales_pct,85.06,84.99,85.20,0.08,0.01,despreciable
app_logins_mes,9.94,10.04,10.00,-0.98,-0.03,despreciable
nps,5.13,5.02,5.00,2.07,0.03,despreciable


#### P5 · d) Distribuciones de métricas palanca

In [40]:
distribuciones("P5", ['ingreso_mensual', 'gasto_mensual', 'pagos_puntuales_pct', 'uso_tarjeta_pct', 'abandono_riesgo'])

#### P5 · d.2) Pool de upgrade Gold → Platinum (CAM009)

Criterio ilustrativo de elegibilidad: cliente **Santander Gold** con ingreso ≥ mediana de los Platinum,
pagos puntuales ≥ 85% y uso de tarjeta ≥ 50%.

In [41]:
gold = clientes.tarjeta_santander == "Santander Gold"
plat = PARTICIONES["P5"]["mask"]
umbral_ing = clientes.loc[plat, "ingreso_mensual"].median()
elegible = gold & (clientes.ingreso_mensual >= umbral_ing) & (clientes.pagos_puntuales_pct >= 85) & (clientes.uso_tarjeta_pct >= 50)
print(f"Gold: {gold.sum():,} | umbral ingreso (mediana Platinum): ${umbral_ing:,.0f} | elegibles upgrade: {elegible.sum():,} "
      f"({elegible.sum()/gold.sum():.1%} de Gold)")
k = campanas.loc[campanas.campaign_id == "CAM009"].iloc[0]
print(f"Con la conversión de CAM009 ({k.conversion_pct}%): ~{elegible.sum()*k.conversion_pct/100:,.0f} upgrades esperados en la muestra")

comp = pd.DataFrame({"Gold elegible": clientes[elegible][VARS_PERFIL].mean(),
                     "Gold no elegible": clientes[gold & ~elegible][VARS_PERFIL].mean(),
                     "Platinum": clientes[plat][VARS_PERFIL].mean()})
display(comp)

Gold: 1,000 | umbral ingreso (mediana Platinum): $26,937 | elegibles upgrade: 118 (11.8% de Gold)
Con la conversión de CAM009 (12%): ~14 upgrades esperados en la muestra


,Gold elegible,Gold no elegible,Platinum
edad,45.81,46.98,46.67
antiguedad_meses,93.37,93.60,93.72
ingreso_mensual,"40,763.78","28,089.10","30,446.16"
uso_tarjeta_pct,73.27,46.04,49.08
transacciones_mes,17.75,18.09,17.95
gasto_mensual,"12,440.51","12,162.80","12,136.82"
ecommerce_pct,53.36,48.51,50.19
pagos_puntuales_pct,92.93,84.46,85.06
app_logins_mes,9.93,10.05,9.94
nps,5.05,4.96,5.13


#### P5 · e) Oportunidades e impacto

In [42]:
op_P5 = oportunidades("P5")
IMPACTOS["P5"] = impacto_estimado("P5")

,clientes,% partición,% base,dif pp
Bajo uso de tarjeta (<25%),238,24.02,24.37,-0.35
Baja activación de promos (<25%),252,25.43,24.49,0.94
Alto riesgo de abandono (≥0.75),269,27.14,24.88,2.26
Detractores NPS (0–6),631,63.67,63.46,0.21
Baja adopción digital (≤6 logins/mes),138,13.93,12.87,1.06
Alto gasto (top 25% base),249,25.13,25.00,0.13
Prioridad retención: alto riesgo + baja activación,79,7.97,6.27,1.70


**Escenario ilustrativo P5** (muestra de 991 clientes, KPIs ponderados de sus campañas): ~242 conversiones (24.4%), gasto medio actual $12,137/mes → incremento estimado de **$518,714/mes** en gasto (+17.7% en convertidos).

**Lectura — P5 Select · Platinum**

- 991 clientes Santander Platinum. ⚠️ Son Premier, no Select.
- Perfil sin diferencias relevantes (|d| ≤ 0.06). Es la partición con **más riesgo relativo**: 27.1% con alto riesgo de abandono (vs 24.9% base) y 8.0% en prioridad de retención (vs 6.3%), aunque la diferencia es pequeña.
- **Upgrade Gold → Platinum (CAM009)**: con un criterio de elegibilidad (ingreso ≥ mediana Platinum, pagos puntuales ≥ 85%, uso ≥ 50%) quedan **118 clientes Gold (11.8%)**, que sí se ven distintos: ingreso ~$40.8k vs ~$28.1k del resto de Gold, uso 73% vs 46%. Con la conversión histórica de 12% serían ~14 upgrades en la muestra; filtrar por elegibilidad debería subir esa conversión frente a la campaña masiva original.
- 2025 (MSI Viajes, 28%) funcionó mucho mejor que 2024 (Upgrade, 12%). Escenario: ~242 conversiones, +$519k/mes.

### 5.6 P6 · Premier · World Elite

Una campaña (CAM020 Beneficios World Elite, 2025), la de menor alcance y mayor Δ gasto relativo.
⚠️ **Inconsistencia de segmento**: la campaña declara Premier, pero **World Elite es 100% Select** en la base.
Es la población más pequeña (302 clientes): las diferencias deben leerse con cautela.

#### P6 · a) Campañas

In [43]:
camp_P6 = tabla_campanas("P6")

,anio,campana,clientes_objetivo,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura
campaign_id,,,,,,,,
CAM020,2025,Beneficios World Elite,12000,71,22,24,12,30.99


#### P6 · b) Población

In [44]:
m = PARTICIONES["P6"]["mask"]
print(f"Clientes en P6: {m.sum():,} ({m.mean():.1%} de la base)")
display(clientes[m].groupby(["tarjeta_santander", "segmento"]).size().rename("clientes").to_frame())

Clientes en P6: 302 (3.0% de la base)


,,clientes
tarjeta_santander,segmento,
World Elite,Select,302


#### P6 · c) Perfil vs resto de la base

In [45]:
perfil_P6 = perfil_vs_resto("P6", etiqueta="P6 · Premier · World Elite")

,media partición,media resto,mediana partición,dif %,d Cohen,magnitud
edad,46.98,46.46,47.50,1.10,0.03,despreciable
antiguedad_meses,93.30,91.61,93.00,1.85,0.03,despreciable
ingreso_mensual,"30,479.53","30,632.06","26,019.24",-0.50,-0.01,despreciable
uso_tarjeta_pct,52.02,49.89,52.70,4.27,0.07,despreciable
transacciones_mes,18.15,17.98,18.00,0.93,0.04,despreciable
gasto_mensual,"11,690.21","12,060.93","10,694.24",-3.07,-0.06,despreciable
ecommerce_pct,48.98,49.67,48.75,-1.39,-0.02,despreciable
pagos_puntuales_pct,85.19,85.00,84.60,0.22,0.02,despreciable
app_logins_mes,9.83,10.04,10.00,-2.05,-0.06,despreciable
nps,4.96,5.03,5.00,-1.46,-0.02,despreciable


#### P6 · d) Distribuciones de métricas palanca

In [46]:
distribuciones("P6", ['gasto_mensual', 'ingreso_mensual', 'uso_tarjeta_pct', 'satisfaccion', 'abandono_riesgo'])

#### P6 · d.2) Robustez: intervalo de confianza de las medias (n = 302)

In [47]:
we = clientes[PARTICIONES["P6"]["mask"]]
ic = pd.DataFrame({"media": we[VARS_PERFIL].mean(),
                   "IC95 inf": we[VARS_PERFIL].mean() - 1.96 * we[VARS_PERFIL].sem(),
                   "IC95 sup": we[VARS_PERFIL].mean() + 1.96 * we[VARS_PERFIL].sem(),
                   "media base": clientes[VARS_PERFIL].mean()})
ic["base dentro del IC"] = ic["media base"].between(ic["IC95 inf"], ic["IC95 sup"])
display(ic)

,media,IC95 inf,IC95 sup,media base,base dentro del IC
edad,46.98,45.03,48.93,46.48,True
antiguedad_meses,93.30,87.56,99.04,91.66,True
ingreso_mensual,"30,479.53","28,751.00","32,208.06","30,627.46",True
uso_tarjeta_pct,52.02,48.83,55.21,49.95,True
transacciones_mes,18.15,17.65,18.64,17.98,True
gasto_mensual,"11,690.21","11,043.02","12,337.40","12,049.73",True
ecommerce_pct,48.98,45.76,52.20,49.65,True
pagos_puntuales_pct,85.19,84.21,86.16,85.00,True
app_logins_mes,9.83,9.47,10.19,10.03,True
nps,4.96,4.60,5.32,5.03,True


#### P6 · e) Oportunidades e impacto

In [48]:
op_P6 = oportunidades("P6")
IMPACTOS["P6"] = impacto_estimado("P6")

,clientes,% partición,% base,dif pp
Bajo uso de tarjeta (<25%),69,22.85,24.37,-1.52
Baja activación de promos (<25%),68,22.52,24.49,-1.97
Alto riesgo de abandono (≥0.75),73,24.17,24.88,-0.71
Detractores NPS (0–6),190,62.91,63.46,-0.55
Baja adopción digital (≤6 logins/mes),47,15.56,12.87,2.69
Alto gasto (top 25% base),69,22.85,25.00,-2.15
Prioridad retención: alto riesgo + baja activación,15,4.97,6.27,-1.30


**Escenario ilustrativo P6** (muestra de 302 clientes, KPIs ponderados de sus campañas): ~66 conversiones (22.0%), gasto medio actual $11,690/mes → incremento estimado de **$186,407/mes** en gasto (+24.0% en convertidos).

**Lectura — P6 Premier · World Elite**

- 302 clientes (3%), la población más pequeña. ⚠️ World Elite es Select en la base, no Premier.
- Para todas las variables, **la media de la base cae dentro del IC 95% de World Elite**: no hay evidencia de un perfil distinto (ingreso igual al de la base, ~$30.5k, poco creíble para una tarjeta de alta gama).
- Su campaña (CAM020) tiene la conversión más baja de las dirigidas (22%) pero el **mayor Δ gasto (24%)**: es la partición con mejor retorno por convertido.
- Señal a vigilar: 15.6% con baja adopción digital (vs 12.9% base). Escenario: ~66 conversiones, +$186k/mes.

## 6. Comparativo entre particiones

In [49]:
d_mat = pd.DataFrame({p: [cohen_d(clientes.loc[v["mask"], c], clientes.loc[~v["mask"], c]) for c in VARS_PERFIL]
                      for p, v in PARTICIONES.items() if p != "P2"}, index=VARS_PERFIL)
fig = px.imshow(d_mat, color_continuous_scale=DIVERGENTE, zmin=-0.5, zmax=0.5, text_auto=".2f", aspect="auto",
                title="d de Cohen de cada partición vs el resto de la base (|d|<0.2 = despreciable)")
fig.update_layout(height=560, coloraxis_colorbar=dict(title="d"))
fig.show()
print("Máx |d| observado:", round(d_mat.abs().max().max(), 3), "→", magnitud(d_mat.abs().max().max()))

Máx |d| observado: 0.078 → despreciable


In [50]:
imp = pd.DataFrame(IMPACTOS.values()).set_index("particion").sort_index()
comparativo = resumen_camp.drop("Total")[["nombre", "campañas", "alcance", "apertura_pct", "conversion_pct",
                                          "incremento_gasto_pct", "incremento_uso_tarjeta_pct", "conv_sobre_apertura"]]
comparativo = comparativo.join(mapa[["segmento consistente", "clientes"]]).join(imp[["incremento_gasto_mes"]])
comparativo["incremento_gasto_por_cliente"] = comparativo.incremento_gasto_mes / comparativo.clientes
display(comparativo.sort_values("incremento_gasto_por_cliente", ascending=False))

,nombre,campañas,alcance,apertura_pct,conversion_pct,incremento_gasto_pct,incremento_uso_tarjeta_pct,conv_sobre_apertura,segmento consistente,clientes,incremento_gasto_mes,incremento_gasto_por_cliente
particion,,,,,,,,,,,,
P2,Todos · Todas,6.00,"2,470,000.00",85.77,39.73,20.72,26.16,46.33,True,10000,"9,922,353.09",992.24
P3,Premier · Aeroméxico Platinum,2.00,"87,000.00",75.55,31.55,17.03,19.52,41.76,True,813,"522,126.09",642.22
P6,Premier · World Elite,1.00,"12,000.00",71.00,22.00,24.00,12.00,30.99,False,302,"186,407.44",617.24
P1,Mass · LikeU,8.00,"1,350,000.00",77.90,34.23,14.19,21.29,43.93,True,2512,"1,473,984.02",586.78
P5,Select · Platinum,2.00,"80,000.00",70.75,24.40,17.68,15.75,34.49,False,991,"518,713.83",523.42
P4,Select · Fiesta Rewards,1.00,"35,000.00",70.00,29.00,14.00,16.00,41.43,False,1206,"584,682.79",484.81


In [51]:
cmp = comparativo.reset_index().sort_values("incremento_gasto_por_cliente")
cmp["etq"] = cmp.particion + " · " + cmp.nombre
fig = go.Figure(go.Bar(y=cmp.etq, x=cmp.incremento_gasto_por_cliente, orientation="h", marker_color=PAL[0],
                       text="$" + cmp.incremento_gasto_por_cliente.round(0).astype(int).astype(str),
                       textposition="outside", cliponaxis=False,
                       hovertemplate="%{y}<br>$%{x:,.0f} por cliente / mes<extra></extra>"))
fig.update_layout(height=340, title="Incremento de gasto esperado por cliente de la partición (escenario ilustrativo, $/mes)",
                  xaxis_title="$ por cliente / mes")
fig.show()

**Lectura — comparativo**

- El mapa de calor lo resume: **ninguna partición se diferencia del resto de la base en ninguna variable** (máx |d| = 0.08). Toda la diferenciación entre particiones está en **cómo responden a las campañas**, no en quiénes son los clientes.
- En gasto incremental esperado por cliente, P2 (masivas) lidera (~$990/mes por cliente), seguida de P3 Aeroméxico (~$640) y P6 World Elite (~$620). P1 LikeU queda a media tabla (~$590) por su bajo Δ gasto, y P4/P5 al final.
- El escenario usa supuestos simples (conversión ponderada de la partición × gasto medio actual × Δ gasto) y sirve para **priorizar**, no como proyección financiera.

## 7. Conclusiones y siguientes pasos

**Conclusiones**

1. **La base de clientes no discrimina entre particiones.** Variables independientes (|ρ| < 0.03) y perfiles idénticos por tarjeta/segmento (|d| < 0.1). En este dataset, segmentar por tarjeta no aporta información de comportamiento; la segmentación útil debe construirse **sobre el comportamiento** (uso, activación de promos, riesgo, adopción digital), que sí tiene varianza dentro de cada partición.
2. **Inconsistencia de segmentos en 3 de 6 particiones** (P4, P5, P6): Premier y Select parecen intercambiados entre el catálogo de campañas y la base. Es el riesgo operativo más importante: filtrar audiencias por `segmento` dejaría fuera a la población correcta.
3. **El desempeño sí varía por partición:** las masivas (P2) y las de cashback con % explícito (P1 2025) son las más eficientes; las premium convierten poco pero, en World Elite, con el mayor Δ gasto.
4. **41.8% de la base (Free, Aeroméxico Blanca, Gold) solo recibe campañas masivas.**
5. **Mecánicas débiles:** Referidos LikeU (CAM010), Reactivación (CAM019) y Upgrade Gold→Platinum (CAM009).

**Siguientes pasos sugeridos**

- Validar con el dueño de los datos el mapeo segmento ↔ tarjeta (Premier/Select) antes de cualquier modelado de audiencias.
- Construir segmentos conductuales dentro de cada partición (ej. clustering sobre uso, activación de promos, riesgo, digital) en lugar de usar la tarjeta como proxy.
- Diseñar campañas dirigidas para Free, Aeroméxico Blanca y Gold (p. ej., upgrade con pool elegible en lugar de masivo).
- Relanzar Fiesta Rewards (P4) con mecánica de % explícito y revisar la propuesta de valor de Reactivación.
- Si existe información de respuesta individual a campañas (quién abrió/convirtió), cruzarla con esta base: con datos agregados por campaña no se puede atribuir respuesta a perfiles de cliente.